In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Load dataset
df = pd.read_csv("uber.csv")

# Handle missing values
df['dropoff_latitude'] = df['dropoff_latitude'].fillna(df['dropoff_latitude'].mean())
df['dropoff_longitude'] = df['dropoff_longitude'].fillna(df['dropoff_longitude'].median())

# Convert pickup_datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Extract date and time features
df['hour'] = df.pickup_datetime.dt.hour
df['day'] = df.pickup_datetime.dt.day
df['month'] = df.pickup_datetime.dt.month
df['year'] = df.pickup_datetime.dt.year
df['dayofweek'] = df.pickup_datetime.dt.dayofweek

df = df.drop('pickup_datetime', axis=1)

# Remove outliers
def remove_outlier(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)
    return df

for col in df.select_dtypes(include=np.number).columns:
    df = remove_outlier(df, col)

# Calculate distance using Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1 = math.radians(lat1), math.radians(lon1)
    lat2, lon2 = math.radians(lat2), math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c

df['dist_travel_km'] = df.apply(
    lambda row: haversine(
        row['pickup_latitude'],
        row['pickup_longitude'],
        row['dropoff_latitude'],
        row['dropoff_longitude']
    ), axis=1
)

# Remove incorrect distances and coordinates
df = df[(df['dist_travel_km'] >= 1) & (df['dist_travel_km'] <= 130)]

df = df[
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]

# Correlation heatmap
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.show()

# Select input and output
x = df[['pickup_longitude', 'pickup_latitude',
        'dropoff_longitude', 'dropoff_latitude',
        'passenger_count', 'hour', 'day', 'month',
        'year', 'dayofweek', 'dist_travel_km']]

y = df['fare_amount']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20
)

# Linear Regression
regression = LinearRegression()
regression.fit(X_train, y_train)

prediction = regression.predict(X_test)

R2_Linear = r2_score(y_test, prediction)
MSE_Linear = mean_squared_error(y_test, prediction)
RMSE_Linear = np.sqrt(MSE_Linear)

print("Linear Regression R2 Score:", R2_Linear)
print("Linear Regression MSE:", MSE_Linear)
print("Linear Regression RMSE:", RMSE_Linear)

# Random Forest Regression
rf = RandomForestRegressor(n_estimators=50)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

R2_Random = r2_score(y_test, y_pred)
MSE_Random = mean_squared_error(y_test, y_pred)
RMSE_Random = np.sqrt(MSE_Random)

print("\nRandom Forest R2 Score:", R2_Random)
print("Random Forest MSE:", MSE_Random)
print("Random Forest RMSE:", RMSE_Random)